# Can a stable finite-element heat step turn a cold node negative?

Start a conducting rod with one positive temperature pulse. Assemble the heat capacity from first principles and determine when an explicit step respects both decay and the sign of every temperature.

## Integrate the heat equation against a hat function

Take a uniform rod of length one with the two endpoint temperatures held at zero. With constant diffusivity $\kappa>0$ the evolution is $u_t=\kappa u_{xx}$. Approximate $u(x,t)$ as a linear combination of continuous piecewise-linear hat functions. Multiply by an interior hat and integrate. Integration by parts transfers one spatial derivative onto each hat. The boundary term vanishes. The nodal temperature vector satisfies $M\dot u=-Ku$.

On an element of length $h$ the two hats are $1-x/h$ and $x/h$ in local coordinates. Integrating their products gives diagonal entries $h/3$ and off-diagonal entries $h/6$ in the element mass matrix. Their constant derivatives give diagonal stiffness $\kappa/h$ and off-diagonal stiffness $-\kappa/h$. After assembly and imposing Dirichlet values, the interior mass has diagonal $4h/6$ and neighbouring entries $h/6$. The stiffness has diagonal $2\kappa/h$ and neighbours $-\kappa/h$.

Forward Euler advances with $G=I-\Delta t M^{-1}K$. On an eigenvector of $K\phi=\lambda M\phi$ it multiplies the amplitude by $1-\Delta t\lambda$. A bound $\Delta t\lambda_{\max}\leq2$ therefore ensures no growth in the mass norm. Preserving nonnegative temperatures for every nonnegative initial vector requires every entry of $G$ to be nonnegative. Each column is the response to a single positive nodal impulse, so the latter condition can be tested directly.

One alternative assigns half of each element's total heat capacity to each endpoint, before removing boundary degrees of freedom. Each interior mass then equals $h$ and the explicit stencil has weights $r,1-2r,r$, with $r=\kappa\Delta t/h^2$. These coefficients are all nonnegative for $0\leq r\leq 1/2$. The first and last interior equations also couple to the fixed zero boundary temperatures.

## Compute the remote temperature after one update

With three interior nodes, $h=1/4$, initially set the left interior node to 1 K above the boundary reference and both others to zero. Inverting the consistent mass in this three-by-three system gives the first column of $(h^2/\kappa)M^{-1}K$ as $(51,-36,9)/14$. At $r=0.1$ the interior temperatures after one step become $(0.63571,0.25714,-0.06429)$ K. The cold node two edges away immediately drops below the lower reference.

The largest dimensionless eigenvalue is about 7.9223. The chosen step gives $r\lambda_{\max}h^2/\kappa\approx0.7922$, comfortably inside the linear stability interval. Lumping instead gives $(0.8,0.1,0)$ K. The negative consistent-mass entry shrinks linearly when the step shrinks. For this impulse its sign persists for every positive first step. With OneSimulation installed locally, the notebook also constructs a physical line mesh and obtains the mass and stiffness by integrating its hat functions at mapped quadrature points. Compare the assembled result against the explicit tridiagonal entries. The experiment shows why reducing the time step can satisfy a stability test without restoring a discrete maximum principle.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
interior_nodes = 3
diffusivity = 1.0
r = 0.10

In [ ]:
assert 3 <= interior_nodes <= 60 and diffusivity > 0 and 0 < r <= 1
n = interior_nodes
h = 1/(n+1)
near = np.diag(np.ones(n-1),1)+np.diag(np.ones(n-1),-1)
mass = h*(4*np.eye(n)+near)/6
stiffness = diffusivity*(2*np.eye(n)-near)/h
dimensionless = h*h*np.linalg.solve(mass,stiffness)/diffusivity
lumped = 2*np.eye(n)-near
u0 = np.zeros(n); u0[0]=1
consistent_u = u0-r*dimensionless@u0
lumped_u = u0-r*lumped@u0
theta = np.pi*np.arange(1,n+1)/(n+1)
eigen = 6*(1-np.cos(theta))/(2+np.cos(theta))
assert np.allclose(np.sort(np.linalg.eigvals(dimensionless)),eigen)
assert np.isclose(consistent_u[2],-r*dimensionless[2,0]) and consistent_u[2]<0
if r<=0.5: assert lumped_u.min() >= -1.e-12
print(f"Time step {r*h*h/diffusivity:.6f}, largest stable r {2/eigen.max():.6f}")
print("First three nodal temperatures:",consistent_u[:3],lumped_u[:3])
try:
    from onesim.cad.d1 import Line
    from onesim.meshing import StructuredLineMesher
    from onesim.numerics.fem.backends.spaces.line_lagrange import LineLagrangeSpace
except ImportError:
    print("The independent OneSimulation element check is available in a local installation.")
else:
    rod = Line(start=(0,0,0),end=(1,0,0))
    grid = StructuredLineMesher(cells=n+1).generate(rod)
    space = LineLagrangeSpace(grid,order=1)
    assembled_mass = np.zeros((n+2,n+2))
    assembled_stiffness = np.zeros_like(assembled_mass)
    for cell,dofs in enumerate(space.cells):
        _,w,(phi,dx,_) = space.element_data(cell)
        local = np.ix_(dofs,dofs)
        assembled_mass[local] += (phi.T*w) @ phi
        assembled_stiffness[local] += diffusivity*(dx.T*w) @ dx
    assert np.allclose(assembled_mass[1:-1,1:-1],mass)
    assert np.allclose(assembled_stiffness[1:-1,1:-1],stiffness)
    print("OneSimulation line-element assembly agrees for both matrices.")

In [ ]:
x = h*np.arange(n+2)
fig, ax = axes("Coordinate along rod", "Temperature above boundary reference (K)")
for values,col,style,label in [(u0,"#747474",":","Initial"),(consistent_u,orange,"-","Consistent mass"),
                                (lumped_u,teal,"--","Lumped mass")]:
    ax.plot(x,np.r_[0.,values,0.],ls=style,marker="o",ms=3,color=col,label=label)
ax.axhline(0,color="#555",lw=.7)
ax.legend()
plt.show()

Line segments are the linear finite-element interpolants. Both ends have a fixed zero temperature and participate in the heat balance.

In [ ]:
fig, ax = axes("Dimensionless time step r", "Remote nodal temperature (K)")
sweep = np.linspace(0,max(.55,1.12*r),201)
ax.plot(sweep,-sweep*dimensionless[2,0],color=orange,label="Third node after one step")
ax.axvline(2/eigen.max(),color="#444",ls=":",label="Stability limit")
ax.scatter(r,consistent_u[2],color=orange)
right = ax.twinx(); right.set_ylabel("Largest mode amplification")
right.plot(sweep,np.max(np.abs(1-sweep[:,None]*eigen),axis=1),color=teal,label="Spectral radius")
right.axhline(1,color=teal,ls="--",lw=.9)
ax.legend(loc="center left",fontsize=8);right.legend(loc="upper center",fontsize=8)
plt.show()

The orange trace is the one-step temperature at the third interior node. The vertical line is the exact stability threshold for this mesh. Crossing it eventually produces amplification greater than one.

## Exercise · keep the lumped temperatures nonnegative

Suppose $h=0.02$ m and $\kappa=8\times10^{-5}$ m²/s. Find the greatest time step allowed by nonnegative weights. What stencil results at that step?

The condition gives $\Delta t=h^2/(2\kappa)=2.5$ s. The central coefficient vanishes and each adjacent value receives weight one half. At the first interior node the fixed boundary value supplies one of these two terms. A step slightly above this threshold gives a negative central coefficient and can drive an isolated positive impulse below zero.

## Exercise · derive the largest mode on successively finer meshes

An interior sine wave has neighbouring sum $2\cos(\theta)$ times its own nodal value. Use this fact to find the limiting stability condition for forward Euler with consistent mass as the number of uniform elements grows.

The mass eigenvalue is $h(2+\cos\theta)/3$ and the stiffness eigenvalue is $2\kappa(1-\cos\theta)/h$. Their ratio, multiplied by $h^2/\kappa$, is $6(1-\cos\theta)/(2+\cos\theta)$. The largest allowed angle tends to $\pi$, giving 12. The limiting stable step therefore has $r\leq1/6$. A positivity test still inspects the entries of the update matrix separately.